In [1]:
# Import libraries
# ==========================================
# Standard Library & File System
# ==========================================
from pathlib import Path
import random
from scipy.stats import chi2_contingency

# ==========================================
# Jupyter Notebook / Display
# ==========================================
from IPython.display import display

# ==========================================
# 2. Data Manipulation
# ==========================================
import numpy as np
import pandas as pd

# ==========================================
# 3. Machine Learning & Preprocessing
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier

# Set random state variable for reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


In [2]:
# Establish paths
project_dir = Path.cwd()
data_path = project_dir / "data" / "student_registry_cleaned_features.csv"
if not data_path.is_file():
    raise FileNotFoundError(
        f"Could not find:\n  {data_path}\n"
        f"Current working directory: {project_dir}\n"
        "Fix: open the Drop_out_predictor folder as the workspace, or run from a terminal: "
        "`cd` to that folder then start Jupyter / run this notebook."
    )

# Load the master processed dataframe
df = pd.read_csv(data_path)

# Ensure original label is numeric (original meaning: graduated=1, dropout=0)
if df["graduated"].dtype == object:
    print("Mapping string target to numeric...")
    df["graduated"] = df["graduated"].map({"graduate": 1, "dropout": 0})

# Modeling target (clear meaning): dropout=1, not_dropout=0
y = (df["graduated"] == 0).astype(int)

print("--- Baseline Evaluation (dropout=1) ---")
print(y.value_counts(normalize=True).rename("proportion"))

# Separate Features (X) and Target (y)
X = df.drop(columns=["graduated"], errors="ignore")
X = X.drop(columns=["student_id"], errors="ignore")

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"\nTraining set size: {X_train.shape[0]} students")
print(f"Testing set size: {X_test.shape[0]} students")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Baseline model: always predicts the most common class
# (Useful to avoid being fooled by high accuracy on imbalanced data)
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
y_dummy_pred = dummy_clf.predict(X_test)

print("\nClassification Report for Dummy (Baseline) Model:")
print(classification_report(y_test, y_dummy_pred, zero_division=0))

--- Baseline Evaluation (dropout=1) ---
graduated
0    0.850946
1    0.149054
Name: proportion, dtype: float64

Training set size: 5920 students
Testing set size: 1480 students
X_train shape: (5920, 25)
X_test shape: (1480, 25)

Classification Report for Dummy (Baseline) Model:
              precision    recall  f1-score   support

           0       0.85      1.00      0.92      1259
           1       0.00      0.00      0.00       221

    accuracy                           0.85      1480
   macro avg       0.43      0.50      0.46      1480
weighted avg       0.72      0.85      0.78      1480



Result: This baseline model always predicts the most common outcome, so while accuracy is high, it does a poor job catching dropouts.

Takeaway: this is the "minimum bar" to beat, and it helps avoid being fooled by high accuracy when the classes are imbalanced.

In [3]:
# Shared column lists (raw features — before any encoding)
# Both Logistic Regression and XGBoost use Pipeline(...); preprocessing fits on train only inside .fit().
cat_cols_raw = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols_raw = [c for c in X_train.columns if c not in cat_cols_raw]
print(f"Categorical columns to one-hot encode: {cat_cols_raw}")

# XGBoost preprocessing: one-hot categoricals, pass through numerics (same idea as before, but used inside a Pipeline below)
preprocess_xgb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_raw),
        ("num", "passthrough", num_cols_raw),
    ],
    verbose_feature_names_out=False,
)

# Class imbalance weight for XGBoost
# With dropout=1, scale_pos_weight should be (non_dropouts / dropouts)
num_dropouts = int(np.sum(y_train == 1))
num_non_dropouts = int(np.sum(y_train == 0))
xgb_scale_pos_weight = num_non_dropouts / max(num_dropouts, 1)
print(f"Calculated scale_pos_weight (non_dropout/dropout): {xgb_scale_pos_weight:.2f}")

Categorical columns to one-hot encode: ['major', 'gender', 'financial_aid', 'tuition_status']
Calculated scale_pos_weight (non_dropout/dropout): 5.71


Result: We define shared raw column lists and an XGBoost preprocessor (`preprocess_xgb`). Both Logistic Regression and XGBoost use **Pipeline** on raw `X_train` / `X_test`, so encoding/scaling is fit only on training data inside `.fit()`. `scale_pos_weight` helps XGBoost emphasize the minority (dropout) class.

In [4]:
# Logistic Regression: same raw column lists; numeric columns scaled (trees use passthrough in preprocess_xgb)
preprocess_lr = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_raw),
        ("num", StandardScaler(), num_cols_raw),
    ],
    verbose_feature_names_out=False,
)

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=5000,
    random_state=RANDOM_STATE,
)

lr_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess_lr),
        ("model", lr_model),
    ]
)

# Train on train only
lr_pipe.fit(X_train, y_train)

# Predict on test
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]  # dropout probability (class 1)

# Report dropout-focused metrics (dropout = 1)
print("--- Logistic Regression Classification Report ---")
print(classification_report(y_test, y_pred_lr, zero_division=0))

print(f"Dropout recall (class 1): {recall_score(y_test, y_pred_lr, pos_label=1):.3f}")
print(f"Dropout precision (class 1): {precision_score(y_test, y_pred_lr, pos_label=1, zero_division=0):.3f}")

cm_lr = confusion_matrix(y_test, y_pred_lr, labels=[0, 1])
print("\nConfusion matrix (rows=true [0,1], cols=pred [0,1]):")
print(cm_lr)

--- Logistic Regression Classification Report ---
              precision    recall  f1-score   support

           0       0.99      0.94      0.96      1259
           1       0.73      0.94      0.82       221

    accuracy                           0.94      1480
   macro avg       0.86      0.94      0.89      1480
weighted avg       0.95      0.94      0.94      1480

Dropout recall (class 1): 0.941
Dropout precision (class 1): 0.730

Confusion matrix (rows=true [0,1], cols=pred [0,1]):
[[1182   77]
 [  13  208]]


Confusion Matrix: 
True negatives- 1182 students who graduated and were predicted to graduate;
False Positive- 77 students who graduated, but were predicted to drop out;
False Negative- 13 students who did not graduate, but were predicted to graduate;
True Positives- 208 students who did not graduate, and were predicted to not graduate.

Result: There were 221 actual dropouts in the test set and this logistic regression model sucessfully flagged 208 of them. The model correctly identified 94.1% of the actual drop-outs, so we can be confident very few students are slipping through the cracks. On the other hand, the precision level of 73% indicates that when the model flags a student as a potential drop-out, it is correct 73% of the time. The other 27% are students flagged as potential drop-outs who actually do go on to graduate (77 false alarms). This precision level could lead to wasted time and effort as well as "alert fatigue", prompting the exploration of a more nuanced model—XGBoost.

In [5]:
# XGBoost: Pipeline on raw X_train / X_test (same pattern as Logistic Regression)
# Preprocessing runs inside the pipeline so it is fit only on training data.
xgb_model = XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=xgb_scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=1,
    eval_metric="logloss",
)

xgb_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess_xgb),
        ("model", xgb_model),
    ]
)

xgb_pipe.fit(X_train, y_train)
y_pred_xgb = xgb_pipe.predict(X_test)

# Evaluate the results
print("\n--- XGBoost Classification Report ---")
print(classification_report(y_test, y_pred_xgb))

# Dropout = 1 (positive class)
dropout_recall = recall_score(y_test, y_pred_xgb, pos_label=1)
dropout_precision = precision_score(y_test, y_pred_xgb, pos_label=1, zero_division=0)
print(f"Dropout recall (class 1): {dropout_recall:.3f}")
print(f"Dropout precision (class 1): {dropout_precision:.3f}")

# Confusion matrix (rows=true, cols=pred) in order [not_dropout(0), dropout(1)]
cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=[0, 1])
print("\nConfusion matrix (rows=true [0,1], cols=pred [0,1]):")
print(cm_xgb)


--- XGBoost Classification Report ---
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1259
           1       0.83      0.85      0.84       221

    accuracy                           0.95      1480
   macro avg       0.90      0.91      0.90      1480
weighted avg       0.95      0.95      0.95      1480

Dropout recall (class 1): 0.846
Dropout precision (class 1): 0.827

Confusion matrix (rows=true [0,1], cols=pred [0,1]):
[[1220   39]
 [  34  187]]


Confusion matrix: True Negative- 1220 students who graduated and were predicted to graduate; False Positive- 39 students who graduated, but were predicted to drop out; False negative- 34 students who were predicted to graduate, but dropped out; True Positive- 187 students who were predicted to drop out and did.

Results: There were 221 actual dropouts in the test set and this XGBoost model sucessfully flagged 187 of them. With rounded percentages, the model correctly identified 85% of the actual drop-outs, meeting the initial goal. The precision level of 83% indicates that when the model flags a student as a potential drop-out, it is correct 83% of the time. The other 17% are students flagged as potential drop-outs who actually do go on to graduate (39 false alarms). This model achieves a balance of recall and precision. By cutting false alarms down to just 39, this model is a great option if intervention resources or staff time are scarce.

In [6]:
# Analyze XGBoost predictions for distribution of errors across majors to ensure overfitting has not occured on a majority class.
pred = y_pred_xgb

eval_df = pd.DataFrame({
    "major": X_test["major"],
    "y_true": y_test,   # dropout=1
    "y_pred": pred,
})
eval_df["wrong"] = eval_df["y_true"] != eval_df["y_pred"]

summary = (
    eval_df.groupby("major")
    .agg(
        n_test=("major", "size"),
        n_wrong=("wrong", "sum"),
    )
)
summary["error_rate"] = (summary["n_wrong"] / summary["n_test"]).round(3)

print("Wrong predictions by major (sorted by count wrong):")
display(summary.sort_values("n_wrong", ascending=False))

print("\nWrong predictions by major (sorted by error rate):")
display(summary.sort_values("error_rate", ascending=False))


Wrong predictions by major (sorted by count wrong):


,n_test,n_wrong,error_rate
major,,,
Computer Science,703,31,0.044
Arts,284,16,0.056
Mathematics,235,16,0.068
Business,258,10,0.039



Wrong predictions by major (sorted by error rate):


,n_test,n_wrong,error_rate
major,,,
Mathematics,235,16,0.068
Arts,284,16,0.056
Computer Science,703,31,0.044
Business,258,10,0.039


Result: Because we had nearly three times as many Computer Science students in our data as any other major, we checked to make sure the model wasn't just guessing well for CS students and failing everyone else. The results show the model performed beautifully across the board. The error rates for all majors were highly consistent, ranging only from 3.9% to 6.8%. This indicates that the model generalized well across different academic disciplines and did not sacrifice minority class performance to favor the majority. 

In [7]:
# Analyze model's ability to predict "Ghost" subgroup on test set
mask = (X_test["is_ghost"] == 1).to_numpy()
y_true_g = y_test.to_numpy()[mask]
y_pred_g = y_pred_xgb[mask]

n_g = int(mask.sum())
print(f"Test rows with is_ghost==1: {n_g}")

if n_g == 0:
    print("No ghost students in test set; skip subgroup metrics.")
else:
    print(f"Dropout recall (ghosts): {recall_score(y_true_g, y_pred_g, pos_label=1):.3f}")
    print(f"Dropout precision (ghosts): {precision_score(y_true_g, y_pred_g, pos_label=1, zero_division=0):.3f}")
    print("\nConfusion matrix [rows=true 0,1 | cols=pred 0,1]:")
    print(confusion_matrix(y_true_g, y_pred_g, labels=[0, 1]))



Test rows with is_ghost==1: 63
Dropout recall (ghosts): 0.982
Dropout precision (ghosts): 0.887

Confusion matrix [rows=true 0,1 | cols=pred 0,1]:
[[ 0  7]
 [ 1 55]]


Result: We specifically tested how well the model predicts outcomes for our "ghost" students—those who are completely digitally disengaged. Out of 63 ghost students in the test set, the model successfully caught 55 out of the 56 students who actually went on to drop out (a 98.2% success rate). It only yielded 7 false alarms. This proves that the model is incredibly reliable at finding at-risk students even when they leave no digital footprint, making it a highly trustworthy tool for targeted outreach.

The "Ghost" Paradox
This subgroup analysis reveals a fascinating paradox regarding digitally disengaged "ghost" students. On paper, ghosts appear academically identical to their active peers, showing no statistically significant differences in prior GPA or credit failure rates. However, their actual outcomes tell a drastically different story: while the overall dropout rate in the test set is just under 15%, a staggering 88.9\% of ghost students (56 out of 63) ended up dropping out. This proves that digital disengagement is one of the strongest behavioral indicators of dropout risk, independent of a student's past academic performance. Because the XGBoost model maintained a 98.2% recall rate on this highly vulnerable group, it stands as an invaluable early-warning tool for counselors to intervene before these silent struggles lead to withdrawal.